# Fase 03 — Split canônico (canonical_split)

**Pergunta:** como garantir que a avaliação reflita zipcode nunca visto?

**Hipótese:** um `train_test_split` aleatório superestima a generalização geográfica, porque o mesmo
zipcode pode aparecer em treino e teste — a validação principal precisa ser por grupo (`zipcode`),
nunca aleatória por linha (P1). Roda depois da fase 02 (EDA completo sobre 100% dos dados) — decisões
descritivas já foram tomadas sem depender do split; agora o split é fixado.

**Módulo:** `src/scripts/create_split.py::run_split_pipeline` (usa
`src/validation/split.py::canonical_split`, `split_summary`; config em `configs/split.yaml`).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.scripts.create_split import run_split_pipeline

metadata = run_split_pipeline()
metadata

{'config': {'split': {'train': 0.7, 'test': 0.15, 'val': 0.15},
  'group_column': 'zipcode',
  'random_state': 42},
 'summary': {'train': {'n_rows': 15100, 'pct_rows': 0.6987, 'n_groups': 49},
  'test': {'n_rows': 2644, 'pct_rows': 0.1223, 'n_groups': 10},
  'val': {'n_rows': 3868, 'pct_rows': 0.179, 'n_groups': 11},
  'group_overlap': [],
  'resales_spanning_splits': 0}}

In [3]:
import pandas as pd

parts_summary = {p: metadata["summary"][p] for p in ["train", "test", "val"]}
summary_df = pd.DataFrame(parts_summary).T
summary_df["group_overlap"] = str(metadata["summary"]["group_overlap"])
summary_df["resales_spanning_splits"] = metadata["summary"]["resales_spanning_splits"]
summary_df

,n_rows,pct_rows,n_groups,group_overlap,resales_spanning_splits
train,15100.0,0.6987,49.0,[],0
test,2644.0,0.1223,10.0,[],0
val,3868.0,0.1790,11.0,[],0


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

s = metadata["summary"]
parts = ["train", "test", "val"]
rows = [s[p]["n_rows"] for p in parts]
groups = [s[p]["n_groups"] for p in parts]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(parts, rows, color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title("linhas por partição")
axes[1].bar(parts, groups, color=["#4C72B0", "#DD8452", "#55A868"])
axes[1].set_title("zipcodes (grupos) por partição")
plt.tight_layout()
plt.savefig("reports/figures/03_split_distribution.png", dpi=110)
plt.close()
print("saved reports/figures/03_split_distribution.png")

saved reports/figures/03_split_distribution.png


## Análise

Split feito com `GroupShuffleSplit` duplo por `zipcode` (nunca aleatório por linha) — ver
`src/validation/split.py::canonical_split`. Percentuais reais variam em torno do alvo (70/15/15) por
causa do tamanho desigual dos zipcodes.

## Resultado (execução real acima)

| Partição | Linhas | % | Zipcodes |
|---|---|---|---|
| train | 15.100 | 69,87% | 49 |
| test | 2.644 | 12,23% | 10 |
| val | 3.868 | 17,90% | 11 |

`group_overlap`: `[]` — nenhum zipcode aparece em mais de uma partição. `resales_spanning_splits`: `0`
(números confirmados pela execução do `metadata` acima, tabela `summary_df`).

![distribuição do split](../reports/figures/03_split_distribution.png)

## Papel de cada partição nesta reestruturação (P1)

`train` ajusta o modelo (e recebe augmentation, se a fase 05 confirmar a hipótese da fase 04). Durante
o desenvolvimento (fases 06-12), `test` é um cheque repetível de "zipcode não visto" — relatado, mas
nunca decide sozinho entre candidatos (decisão sempre via `GroupKFold` dentro de `train`). `val` só é
tocado **uma única vez**, na fase 13, depois de tudo travado.

## Decisão

Split aceito. Segue para fase 04 (cobertura geográfica).

## Artefatos

`data/processed/split_assignment.parquet`, `data/processed/split_metadata.json`,
`reports/figures/03_split_distribution.png`